# 📒 Susu Books — Fine-Tuning Gemma 4 E4B for Multilingual Transaction Extraction

**Gemma 4 Good Hackathon | Kaggle 2026**

This notebook fine-tunes **Gemma 4 E4B** (4-billion-parameter edge model) using **Unsloth** and LoRA
to improve structured transaction extraction from natural conversational speech across five African languages.

### What we train
A market trader says: *"Metɔɔ shinkafa bags 3 a GHS 150 koro koro wɔ Kofi hɔ"* (Twi)
The model must call: `record_purchase(item="rice", quantity=3, unit_price=150, unit="bags", supplier="Kofi", currency="GHS")`

### Training plan
| | |
|---|---|
| **Base model** | `google/gemma-4-e4b-it` via Unsloth |
| **Method** | LoRA (r=16, α=32) |
| **Data** | 1 800 train + 200 validation synthetic examples |
| **Languages** | English · Twi · Hausa · Pidgin English · Swahili |
| **Functions** | `record_purchase` · `record_sale` · `record_expense` |
| **Epochs** | 3 |
| **Hardware** | Kaggle T4 GPU (16 GB) |
| **Output** | LoRA adapter + merged GGUF for Ollama |


## 1. Environment Setup

In [ ]:
%%capture
# Unsloth — efficient LoRA fine-tuning with 2x speed, 60% less memory
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps trl peft accelerate bitsandbytes
!pip install datasets evaluate

In [ ]:
import json
import random
import re
import os
import math
from collections import defaultdict
from typing import Any

import torch
from datasets import Dataset, DatasetDict
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel, is_bfloat16_supported

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU      : {torch.cuda.get_device_name(0)}")
    print(f"VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
random.seed(42)

## 2. Configuration

In [ ]:
# ── Model ──────────────────────────────────────────────────────────────────
MODEL_NAME      = "unsloth/gemma-4-e4b-it"   # Unsloth-optimised checkpoint
MAX_SEQ_LENGTH  = 2048
LOAD_IN_4BIT    = True

# ── LoRA ───────────────────────────────────────────────────────────────────
LORA_RANK       = 16
LORA_ALPHA      = 32
LORA_DROPOUT    = 0.05

# ── Training ───────────────────────────────────────────────────────────────
NUM_EPOCHS        = 3
BATCH_SIZE        = 4
GRAD_ACCUM        = 4          # effective batch = 16
LEARNING_RATE     = 2e-4
WARMUP_RATIO      = 0.05
LR_SCHEDULER      = "cosine"
MAX_GRAD_NORM     = 1.0
WEIGHT_DECAY      = 0.01

# ── Data ───────────────────────────────────────────────────────────────────
TOTAL_EXAMPLES  = 2000
VAL_EXAMPLES    = 200
TRAIN_EXAMPLES  = TOTAL_EXAMPLES - VAL_EXAMPLES

# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_DIR      = "/kaggle/working/susu-books-lora"
MERGED_DIR      = "/kaggle/working/susu-books-merged"
GGUF_QUANT      = "q4_k_m"    # Ollama-friendly quantisation

## 3. Susu Books System Prompt & Function Schemas

In [ ]:
SYSTEM_PROMPT = """You are Susu Books, an AI business copilot for market traders and informal economy workers in Africa. You help record financial transactions — purchases, sales, and expenses — by calling the appropriate function.

When the user describes a transaction in ANY language (English, Twi, Hausa, Pidgin English, Swahili, or a mix), extract the key details and call the correct function. Always call a function — never just reply with text for transaction descriptions.

Rules:
- Extract numeric quantities accurately (words like "three" = 3, "dozen" = 12)
- Default currency: GHS (Ghana cedis) unless another is clear from context
- If unit is unclear, use the most natural one for that item (rice → bags, onions → kg, palm oil → liters)
- item names should be in English, lowercase, snake_case (e.g. palm_oil, rice, tomatoes)
- For expenses, category can be: transport, market_fee, utilities, labour, packaging, other
"""

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "record_purchase",
            "description": "Record buying stock or supplies from a supplier.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item":       {"type": "string",  "description": "Item name in English, snake_case"},
                    "quantity":   {"type": "number",  "description": "How many units purchased"},
                    "unit":       {"type": "string",  "description": "Unit of measure (bags, kg, liters, crates, pieces, bunches, tubers, baskets, tins)"},
                    "unit_price": {"type": "number",  "description": "Price per unit"},
                    "total_amount":{"type": "number", "description": "Total paid (optional if unit_price given)"},
                    "supplier":   {"type": "string",  "description": "Name of supplier (optional)"},
                    "currency":   {"type": "string",  "description": "Currency code: GHS, NGN, KES, XOF"},
                },
                "required": ["item", "quantity", "unit", "currency"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "record_sale",
            "description": "Record selling goods to a customer.",
            "parameters": {
                "type": "object",
                "properties": {
                    "item":       {"type": "string",  "description": "Item name in English, snake_case"},
                    "quantity":   {"type": "number",  "description": "How many units sold"},
                    "unit":       {"type": "string",  "description": "Unit of measure"},
                    "unit_price": {"type": "number",  "description": "Price per unit"},
                    "total_amount":{"type": "number", "description": "Total received"},
                    "customer":   {"type": "string",  "description": "Customer name (optional)"},
                    "currency":   {"type": "string",  "description": "Currency code"},
                },
                "required": ["item", "quantity", "unit", "currency"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "record_expense",
            "description": "Record a business expense (transport, market fee, utilities, etc).",
            "parameters": {
                "type": "object",
                "properties": {
                    "description": {"type": "string", "description": "What the expense was for"},
                    "amount":      {"type": "number", "description": "Amount paid"},
                    "category":    {"type": "string", "description": "Category: transport, market_fee, utilities, labour, packaging, other"},
                    "currency":    {"type": "string", "description": "Currency code"},
                },
                "required": ["description", "amount", "category", "currency"]
            }
        }
    }
]

print("System prompt word count:", len(SYSTEM_PROMPT.split()))
print("Functions defined:", [t["function"]["name"] for t in TOOLS])

## 4. Synthetic Training Data Generation

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Reference data: items, suppliers, customers, currencies
# ──────────────────────────────────────────────────────────────────────────

ITEMS = {
    "rice":        {"units": ["bags", "kg"],           "currencies": ["GHS", "NGN"]},
    "beans":       {"units": ["bags", "kg"],           "currencies": ["GHS", "NGN"]},
    "palm_oil":    {"units": ["liters", "tins"],       "currencies": ["GHS", "NGN", "XOF"]},
    "tomatoes":    {"units": ["crates", "kg"],         "currencies": ["GHS", "NGN"]},
    "onions":      {"units": ["kg", "bags"],           "currencies": ["GHS", "NGN"]},
    "plantains":   {"units": ["bunches", "kg"],        "currencies": ["GHS", "XOF"]},
    "fish":        {"units": ["kg", "pieces"],         "currencies": ["GHS", "NGN", "KES"]},
    "sugar":       {"units": ["kg", "bags"],           "currencies": ["GHS", "NGN", "KES"]},
    "flour":       {"units": ["bags", "kg"],           "currencies": ["GHS", "NGN"]},
    "salt":        {"units": ["bags", "kg", "tins"],   "currencies": ["GHS", "NGN", "KES"]},
    "yams":        {"units": ["tubers", "kg"],         "currencies": ["GHS", "NGN"]},
    "cassava":     {"units": ["kg", "baskets"],        "currencies": ["GHS", "NGN"]},
    "groundnuts":  {"units": ["kg", "bags"],           "currencies": ["GHS", "NGN"]},
    "garri":       {"units": ["kg", "bags"],           "currencies": ["NGN", "GHS"]},
    "kenkey":      {"units": ["pieces", "kg"],         "currencies": ["GHS"]},
    "maize":       {"units": ["bags", "kg"],           "currencies": ["GHS", "NGN", "KES"]},
    "cooking_oil": {"units": ["liters", "tins"],       "currencies": ["GHS", "NGN", "KES"]},
    "pepper":      {"units": ["kg", "crates"],         "currencies": ["GHS", "NGN"]},
}

GH_SUPPLIERS  = ["Kofi", "Ama", "Kwame", "Abena", "Yaw", "Akosua", "Kojo", "Adwoa"]
NG_SUPPLIERS  = ["Emeka", "Ngozi", "Chidi", "Amaka", "Ibrahim", "Fatima", "Musa"]
KE_SUPPLIERS  = ["Wanjiku", "Kamau", "Akinyi", "Omondi", "Fatuma", "Hassan"]
ALL_SUPPLIERS = GH_SUPPLIERS + NG_SUPPLIERS + KE_SUPPLIERS

GH_CUSTOMERS  = ["Maame", "Auntie Grace", "Sister Ama", "Nana", "Abena", "Mrs Adjei"]
NG_CUSTOMERS  = ["Mama Ngozi", "Iya Beji", "Auntie Ngozi", "Sister Chioma", "Hajiya"]
KE_CUSTOMERS  = ["Mama mboga", "Mama Wanjiku", "Bibi", "Shangazi"]
ALL_CUSTOMERS = GH_CUSTOMERS + NG_CUSTOMERS + KE_CUSTOMERS

EXPENSE_TYPES = {
    "transport":   ["trotro fare", "transport to market", "bus fare", "okada", "boda boda",
                    "taxi to market", "lorry fare", "motorbike", "keke napep"],
    "market_fee":  ["market stall fee", "daily market levy", "market dues", "stall fee",
                    "toll fee", "market tax", "daily stall", "GES levy"],
    "utilities":   ["electricity bill", "phone credit", "mobile data", "generator fuel",
                    "water bill", "phone charging"],
    "labour":      ["helper wages", "porter fee", "loading fee", "carrier fee"],
    "packaging":   ["bags", "nylon bags", "rubber bands", "packaging materials", "polythene"],
    "other":       ["cleaning supplies", "scale repair", "table rental", "signboard"],
}

# Local item name variations per language
ITEM_LOCAL = {
    "rice": {
        "tw": ["shinkafa", "ɛnam-ɛnam", "rice"],
        "ha": ["shinkafa", "shinkafar", "rice"],
        "pcm": ["rice", "rais"],
        "sw": ["mchele", "wali", "rice"],
        "en": ["rice"],
    },
    "tomatoes": {
        "tw": ["tomatoes", "tomato", "ntɔmatɔ"],
        "ha": ["tumatur", "tumatir", "tomato"],
        "pcm": ["tomatoes", "tomato", "tomates"],
        "sw": ["nyanya", "nyanyaa", "tomato"],
        "en": ["tomatoes"],
    },
    "onions": {
        "tw": ["onions", "gyeene", "onion"],
        "ha": ["albasa", "albasaa", "onion"],
        "pcm": ["onions", "onion", "oniyọ̃"],
        "sw": ["vitunguu", "kitunguu"],
        "en": ["onions"],
    },
    "palm_oil": {
        "tw": ["nkuto", "palm oil", "nkuta"],
        "ha": ["mai ja", "mai-ja", "palm oil"],
        "pcm": ["palm oil", "red oil", "palm ọil"],
        "sw": ["mafuta ya mawese", "mafuta ya nazi"],
        "en": ["palm oil"],
    },
    "plantains": {
        "tw": ["aborɔdze", "plantains", "aburɔdze"],
        "ha": ["ayaba", "plantain"],
        "pcm": ["plantain", "plantains", "unripe banana"],
        "sw": ["ndizi", "ndizi mbivu"],
        "en": ["plantains"],
    },
    "fish": {
        "tw": ["nsuomnam", "fish", "koobi"],
        "ha": ["kifi", "fish", "kifiyo"],
        "pcm": ["fish", "eja", "freshfish"],
        "sw": ["samaki", "dagaa"],
        "en": ["fish"],
    },
    "yams": {
        "tw": ["ɛde", "yam", "yams"],
        "ha": ["doya", "yams"],
        "pcm": ["yam", "yams", "iyan"],
        "sw": ["viazi vikuu", "muhogo"],
        "en": ["yams"],
    },
    "beans": {
        "tw": ["adua", "beans"],
        "ha": ["wake", "waken", "beans"],
        "pcm": ["beans", "ewa"],
        "sw": ["maharagwe", "choroko"],
        "en": ["beans"],
    },
}

# Quantity-as-words (for realism)
NUMBERS_EN  = {1:"one",2:"two",3:"three",4:"four",5:"five",6:"six",7:"seven",
               8:"eight",9:"nine",10:"ten",12:"twelve",15:"fifteen",20:"twenty",
               25:"twenty-five",50:"fifty"}
NUMBERS_TW  = {1:"baako",2:"abien",3:"abiɛsa",4:"anan",5:"enum",
               6:"asia",7:"ason",8:"awɔtwe",9:"akron",10:"du"}
NUMBERS_HA  = {1:"ɗaya",2:"biyu",3:"uku",4:"huɗu",5:"biyar",
               6:"shida",7:"bakwai",8:"takwas",9:"tara",10:"goma"}
NUMBERS_SW  = {1:"moja",2:"mbili",3:"tatu",4:"nne",5:"tano",
               6:"sita",7:"saba",8:"nane",9:"tisa",10:"kumi"}

def rand_qty():
    """Return a realistic quantity."""
    weights = [3,3,3,2,2,2,1,1,1,1]
    choices = [1,2,3,4,5,6,8,10,12,20]
    return random.choices(choices, weights=weights)[0]

def rand_price(item: str, unit: str) -> float:
    """Return a realistic price for item/unit combo."""
    ranges = {
        ("rice","bags"):        (100, 180),
        ("rice","kg"):          (3,   7),
        ("beans","bags"):       (90,  160),
        ("beans","kg"):         (4,   9),
        ("palm_oil","liters"):  (25,  70),
        ("palm_oil","tins"):    (200, 450),
        ("tomatoes","crates"):  (40,  120),
        ("tomatoes","kg"):      (5,   15),
        ("onions","kg"):        (4,   15),
        ("onions","bags"):      (60,  120),
        ("plantains","bunches"):(15,  50),
        ("plantains","kg"):     (2,   8),
        ("fish","kg"):          (30,  80),
        ("fish","pieces"):      (5,   25),
        ("sugar","kg"):         (6,   12),
        ("sugar","bags"):       (50,  120),
        ("flour","bags"):       (80,  160),
        ("salt","kg"):          (2,   6),
        ("salt","bags"):        (30,  80),
        ("yams","tubers"):      (5,   20),
        ("yams","kg"):          (3,   10),
        ("cassava","kg"):       (2,   6),
        ("cassava","baskets"):  (30,  80),
        ("groundnuts","kg"):    (8,   20),
        ("maize","bags"):       (60,  130),
        ("cooking_oil","liters"):(25, 65),
        ("pepper","kg"):        (15,  40),
    }
    lo, hi = ranges.get((item, unit), (10, 200))
    price = random.uniform(lo, hi)
    # Round to nearest 5 or 10 for realism
    if price > 50:
        price = round(price / 5) * 5
    else:
        price = round(price * 2) / 2
    return price

print("Data reference tables loaded.")
print(f"  Items: {len(ITEMS)}")
print(f"  Suppliers: {len(ALL_SUPPLIERS)}")
print(f"  Customers: {len(ALL_CUSTOMERS)}")
print(f"  Expense categories: {len(EXPENSE_TYPES)}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Template generators per language
# ──────────────────────────────────────────────────────────────────────────

def get_local_item(item: str, lang: str) -> str:
    """Get a local-language name for an item (with fallback to English)."""
    variants = ITEM_LOCAL.get(item, {}).get(lang, [item.replace("_", " ")])
    return random.choice(variants)

def num_as_word(n: int, lang: str) -> str:
    """Sometimes return the number as a word in the target language."""
    if random.random() < 0.25:
        table = {"en": NUMBERS_EN, "tw": NUMBERS_TW, "ha": NUMBERS_HA, "sw": NUMBERS_SW}
        t = table.get(lang, NUMBERS_EN)
        if n in t:
            return t[n]
    return str(n)

def currency_spoken(currency: str, lang: str) -> str:
    """How a currency is spoken in context."""
    spoken = {
        "GHS": {"en": ["cedis", "Ghana cedis", "GHS"], "tw": ["cedis", "sika", "GHS"],
                "ha": ["cedis", "GHS"], "pcm": ["cedis", "Ghana cedis"], "sw": ["cedis"]},
        "NGN": {"en": ["naira", "NGN", "naira each"], "tw": ["naira"], "ha": ["naira", "NGN"],
                "pcm": ["naira", "naira"], "sw": ["naira"]},
        "KES": {"en": ["shillings", "KES", "Kenyan shillings"],
                "sw": ["shilingi", "KES", "shillings"],
                "ha": ["shillings"], "pcm": ["shillings"], "tw": ["shillings"]},
        "XOF": {"en": ["CFA francs", "CFA", "XOF"], "tw": ["CFA"], "ha": ["CFA"],
                "pcm": ["CFA", "franc"], "sw": ["CFA", "franc"]},
    }
    options = spoken.get(currency, {}).get(lang, [currency])
    return random.choice(options)

# ──────────────────────────────────────────────────────────────────────────
# Purchase templates
# ──────────────────────────────────────────────────────────────────────────

def purchase_en(item, qty, unit, price, total, supplier, currency):
    i = item.replace("_", " ")
    c = currency_spoken(currency, "en")
    sup = f" from {supplier}" if supplier and random.random() > 0.4 else ""
    templates = [
        f"I bought {qty} {unit} of {i}{sup} for {price} {c} each",
        f"Purchased {qty} {unit} of {i} at {price} {c}{sup}",
        f"Got {qty} {unit} {i}{sup}, paid {price} {c} per {unit.rstrip('s')}",
        f"{supplier + ' sold me' if supplier else 'Bought'} {qty} {unit} of {i} for {total:.0f} {c} total",
        f"I bought {num_as_word(qty,'en')} {unit} of {i}{sup} — {price} {c} each",
        f"Stock purchase: {qty} {unit} {i}, {price} {c}/{unit.rstrip('s')}{sup}",
        f"Restock — {i}, {qty} {unit}{sup}, {price} {c} each",
        f"Just bought {qty} {unit} of {i}{sup}. {price} {c} per {unit.rstrip('s')}.",
        f"I paid {total:.0f} {c} for {qty} {unit} of {i}{sup}",
    ]
    return random.choice(templates)

def purchase_tw(item, qty, unit, price, total, supplier, currency):
    i = get_local_item(item, "tw")
    c = currency_spoken(currency, "tw")
    sup = f" wɔ {supplier} hɔ" if supplier and random.random() > 0.5 else ""
    q = num_as_word(qty, "tw")
    templates = [
        f"Metɔɔ {i} {unit} {q} a {price} {c} koro koro{sup}",
        f"Wɔatɔ {i} {q} {unit} {price} {c} biara{sup}",
        f"Metɔɔ {i} wɔ{' ' + supplier if supplier else ''} — {q} {unit}, {price} {c} biara",
        f"Me tɔɔ {i} {q} {unit} ntɛm yi{sup}. {price} {c} biara.",
        f"Mede {total:.0f} {c} tɔɔ {i} {q} {unit}{sup}",
        f"Atɔ {i}: {q} {unit}, {price} {c}/{unit.rstrip('s')}{sup}",
    ]
    return random.choice(templates)

def purchase_ha(item, qty, unit, price, total, supplier, currency):
    i = get_local_item(item, "ha")
    c = currency_spoken(currency, "ha")
    sup = f" daga {supplier}" if supplier and random.random() > 0.5 else ""
    q = num_as_word(qty, "ha")
    templates = [
        f"Na saya {i} {q} {unit}{sup} da {price} {c} kowane",
        f"Na je kasuwa na sayi {i} {q} {unit}{sup} zuwa {price} {c}",
        f"Na sayo {i}{sup} — {q} {unit}, {price} {c} kowane",
        f"Siyan {i} {q} {unit}, {price} {c} ɗaya{sup}",
        f"Na biya {total:.0f} {c} don {i} {q} {unit}{sup}",
        f"Kasuwa: na sayi {i} {q} {unit} a {price} {c}{sup}",
    ]
    return random.choice(templates)

def purchase_pcm(item, qty, unit, price, total, supplier, currency):
    i = item.replace("_", " ")
    c = currency_spoken(currency, "pcm")
    sup = f" from {supplier}" if supplier and random.random() > 0.4 else ""
    templates = [
        f"I don buy {qty} {unit} {i}{sup} for {price} {c}",
        f"Na {price} {c} I pay for {qty} {unit} {i}{sup}",
        f"I buy {i} {qty} {unit}{sup} — {price} {c} each one",
        f"I go market go buy {qty} {unit} {i}{sup}, e cost {price} {c} per {unit.rstrip('s')}",
        f"I pay {total:.0f} {c} for {qty} {unit} {i}{sup}",
        f"Buy {i}: {qty} {unit}, {price} {c} each{sup}",
        f"I don get {qty} {unit} {i}{sup} for {price} {c} per piece",
    ]
    return random.choice(templates)

def purchase_sw(item, qty, unit, price, total, supplier, currency):
    i = get_local_item(item, "sw")
    c = currency_spoken(currency, "sw")
    sup = f" kutoka {supplier}" if supplier and random.random() > 0.5 else ""
    q = num_as_word(qty, "sw")
    templates = [
        f"Nilinunua {i} {q} {unit}{sup} kwa {price} {c}",
        f"Nimeamua kununua {q} {unit} za {i}{sup}, bei {price} {c} kila kimoja",
        f"Nmenunua {i} {q} {unit}{sup} — {price} {c} moja moja",
        f"Nimepata {q} {unit} {i}{sup} kwa {total:.0f} {c} jumla",
        f"Soko: {i} {q} {unit}, {price} {c} kila {unit.rstrip('s')}{sup}",
        f"Nilipenda kununua {i} {q} {unit}{sup} kwa bei ya {price} {c}",
    ]
    return random.choice(templates)

# ──────────────────────────────────────────────────────────────────────────
# Sale templates
# ──────────────────────────────────────────────────────────────────────────

def sale_en(item, qty, unit, price, total, customer, currency):
    i = item.replace("_", " ")
    c = currency_spoken(currency, "en")
    cust = f" to {customer}" if customer and random.random() > 0.4 else ""
    templates = [
        f"Sold {qty} {unit} of {i}{cust} for {price} {c} each",
        f"I sold {i} — {qty} {unit}{cust}, {price} {c} per {unit.rstrip('s')}",
        f"{customer + ' bought' if customer else 'Sold'} {qty} {unit} of {i} for {total:.0f} {c}",
        f"Just sold {qty} {unit} {i}{cust} at {price} {c}",
        f"Sale: {qty} {unit} {i}{cust}, {price} {c} each",
        f"I made a sale — {qty} {unit} {i}, {price} {c}/{unit.rstrip('s')}{cust}",
        f"Sold {num_as_word(qty,'en')} {unit} of {i}{cust} at {price} {c} each",
        f"{qty} {unit} {i} gone{cust}, paid {price} {c} per {unit.rstrip('s')}",
    ]
    return random.choice(templates)

def sale_tw(item, qty, unit, price, total, customer, currency):
    i = get_local_item(item, "tw")
    c = currency_spoken(currency, "tw")
    cust = f" kyɛɛ {customer}" if customer and random.random() > 0.5 else ""
    q = num_as_word(qty, "tw")
    templates = [
        f"Mitan {i} {q} {unit}{cust} si {price} {c} biara",
        f"Me tɔɔ {i}{cust} — {q} {unit}, {price} {c} koro koro",
        f"Wɔatɔ {i} {q} {unit}{cust} si {price} {c}",
        f"Asale a: {i} {q} {unit}, {price} {c}{cust}",
        f"Mede {total:.0f} {c} gye oo {i} {q} {unit}{cust}",
    ]
    return random.choice(templates)

def sale_ha(item, qty, unit, price, total, customer, currency):
    i = get_local_item(item, "ha")
    c = currency_spoken(currency, "ha")
    cust = f" ga {customer}" if customer and random.random() > 0.5 else ""
    q = num_as_word(qty, "ha")
    templates = [
        f"Na sayar da {i} {q} {unit}{cust} da {price} {c} kowane",
        f"Na iyar da {i} {q} {unit}{cust} zuwa {price} {c}",
        f"Sayarwa: {i} {q} {unit}, {price} {c}/{unit.rstrip('s')}{cust}",
        f"Na bada {q} {unit} {i}{cust} da {price} {c} ɗaya",
        f"Na karɓa {total:.0f} {c} daga sayar {i} {q} {unit}{cust}",
    ]
    return random.choice(templates)

def sale_pcm(item, qty, unit, price, total, customer, currency):
    i = item.replace("_", " ")
    c = currency_spoken(currency, "pcm")
    cust = f" to {customer}" if customer and random.random() > 0.4 else ""
    templates = [
        f"I sell {qty} {unit} {i}{cust} for {price} {c}",
        f"I don sell {i} {qty} {unit}{cust} — {price} {c} each",
        f"{customer + ' take' if customer else 'Person take'} {qty} {unit} {i}, I collect {price} {c}",
        f"Sale: {qty} {unit} {i}{cust}, {price} {c} per unit",
        f"I give {qty} {unit} {i}{cust} for {total:.0f} {c} total",
        f"Na sell {qty} {unit} {i}{cust} for {price} {c} each one",
    ]
    return random.choice(templates)

def sale_sw(item, qty, unit, price, total, customer, currency):
    i = get_local_item(item, "sw")
    c = currency_spoken(currency, "sw")
    cust = f" kwa {customer}" if customer and random.random() > 0.5 else ""
    q = num_as_word(qty, "sw")
    templates = [
        f"Niliuza {i} {q} {unit}{cust} kwa {price} {c}",
        f"Nimemuuzia {customer if customer else 'mteja'} {q} {unit} {i} bei {price} {c}",
        f"Uuzaji: {i} {q} {unit}, {price} {c} kila kimoja{cust}",
        f"Nimepokea {total:.0f} {c} kwa {q} {unit} {i}{cust}",
        f"Niliuza {i}{cust} — {q} {unit} kwa {price} {c} kila moja",
    ]
    return random.choice(templates)

# ──────────────────────────────────────────────────────────────────────────
# Expense templates
# ──────────────────────────────────────────────────────────────────────────

def expense_en(desc, amount, category, currency):
    c = currency_spoken(currency, "en")
    templates = [
        f"Paid {amount} {c} for {desc}",
        f"{desc} cost me {amount} {c} today",
        f"I spent {amount} {c} on {desc}",
        f"Expense: {desc}, {amount} {c}",
        f"{amount} {c} for {desc} today",
        f"Business expense — {desc}: {amount} {c}",
        f"I paid {amount} {c} for {desc} this morning",
    ]
    return random.choice(templates)

def expense_tw(desc, amount, category, currency):
    c = currency_spoken(currency, "tw")
    templates = [
        f"Mepae {amount} {c} ma {desc}",
        f"{desc} kɛ me {amount} {c} ɛnnɛ",
        f"Mede {amount} {c} bɔɔ {desc}",
        f"Mepaa sika {amount} {c} wɔ {desc} ho",
        f"Ahoden: {desc}, {amount} {c}",
    ]
    return random.choice(templates)

def expense_ha(desc, amount, category, currency):
    c = currency_spoken(currency, "ha")
    templates = [
        f"Na biya {amount} {c} don {desc}",
        f"{desc} ya kai {amount} {c} yau",
        f"Na kashe {amount} {c} a kan {desc}",
        f"Kashe kuɗi: {desc}, {amount} {c}",
        f"Na biya {amount} {c} na {desc} a yau",
    ]
    return random.choice(templates)

def expense_pcm(desc, amount, category, currency):
    c = currency_spoken(currency, "pcm")
    templates = [
        f"I pay {amount} {c} for {desc}",
        f"{desc} cost {amount} {c}",
        f"I spend {amount} {c} on {desc} today",
        f"Business expense: {desc}, {amount} {c}",
        f"Na pay {amount} {c} for {desc} today morning",
        f"{amount} {c} don go for {desc}",
    ]
    return random.choice(templates)

def expense_sw(desc, amount, category, currency):
    c = currency_spoken(currency, "sw")
    templates = [
        f"Nililipa {amount} {c} kwa {desc}",
        f"{desc} ilinigharimu {amount} {c} leo",
        f"Nimetumia {amount} {c} kwa {desc}",
        f"Gharama: {desc}, {amount} {c}",
        f"Nilipanda {amount} {c} kwa {desc} asubuhi",
    ]
    return random.choice(templates)

LANG_PURCHASE = {"en": purchase_en, "tw": purchase_tw, "ha": purchase_ha,
                 "pcm": purchase_pcm, "sw": purchase_sw}
LANG_SALE     = {"en": sale_en, "tw": sale_tw, "ha": sale_ha,
                 "pcm": sale_pcm, "sw": sale_sw}
LANG_EXPENSE  = {"en": expense_en, "tw": expense_tw, "ha": expense_ha,
                 "pcm": expense_pcm, "sw": expense_sw}

print("Template generators ready for 5 languages × 3 transaction types.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Generate one example of each type
# ──────────────────────────────────────────────────────────────────────────

def make_purchase(lang: str, currency_override: str | None = None) -> dict:
    item     = random.choice(list(ITEMS.keys()))
    info     = ITEMS[item]
    unit     = random.choice(info["units"])
    currency = currency_override or random.choice(info["currencies"])
    qty      = rand_qty()
    price    = rand_price(item, unit)
    total    = qty * price
    supplier = random.choice(ALL_SUPPLIERS) if random.random() > 0.35 else None

    utterance = LANG_PURCHASE[lang](item, qty, unit, price, total, supplier, currency)

    args: dict[str, Any] = {
        "item": item,
        "quantity": qty,
        "unit": unit,
        "unit_price": price,
        "currency": currency,
    }
    if supplier:
        args["supplier"] = supplier

    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": utterance},
            {"role": "assistant", "content": None,
             "tool_calls": [{"type": "function",
                             "function": {"name": "record_purchase", "arguments": args}}]},
        ],
        "_meta": {"lang": lang, "type": "purchase", "item": item}
    }


def make_sale(lang: str, currency_override: str | None = None) -> dict:
    item     = random.choice(list(ITEMS.keys()))
    info     = ITEMS[item]
    unit     = random.choice(info["units"])
    currency = currency_override or random.choice(info["currencies"])
    qty      = rand_qty()
    price    = rand_price(item, unit) * random.uniform(1.1, 1.45)  # sell above cost
    price    = round(price / 5) * 5 if price > 30 else round(price * 2) / 2
    total    = qty * price
    customer = random.choice(ALL_CUSTOMERS) if random.random() > 0.4 else None

    utterance = LANG_SALE[lang](item, qty, unit, price, total, customer, currency)

    args: dict[str, Any] = {
        "item": item,
        "quantity": qty,
        "unit": unit,
        "unit_price": price,
        "currency": currency,
    }
    if customer:
        args["customer"] = customer

    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": utterance},
            {"role": "assistant", "content": None,
             "tool_calls": [{"type": "function",
                             "function": {"name": "record_sale", "arguments": args}}]},
        ],
        "_meta": {"lang": lang, "type": "sale", "item": item}
    }


def make_expense(lang: str, currency_override: str | None = None) -> dict:
    # Match currency to language region
    if currency_override:
        currency = currency_override
    elif lang in ("tw", "pcm"):
        currency = random.choice(["GHS", "NGN"])
    elif lang == "sw":
        currency = "KES"
    elif lang == "ha":
        currency = random.choice(["NGN", "GHS"])
    else:
        currency = random.choice(["GHS", "NGN", "KES"])

    category = random.choice(list(EXPENSE_TYPES.keys()))
    desc     = random.choice(EXPENSE_TYPES[category])
    amount   = round(random.choice([5,8,10,12,15,20,25,30,40,50,60,80,100,120,150,200]))

    utterance = LANG_EXPENSE[lang](desc, amount, category, currency)

    args: dict[str, Any] = {
        "description": desc,
        "amount": amount,
        "category": category,
        "currency": currency,
    }

    return {
        "messages": [
            {"role": "system",    "content": SYSTEM_PROMPT},
            {"role": "user",      "content": utterance},
            {"role": "assistant", "content": None,
             "tool_calls": [{"type": "function",
                             "function": {"name": "record_expense", "arguments": args}}]},
        ],
        "_meta": {"lang": lang, "type": "expense"}
    }


# ──────────────────────────────────────────────────────────────────────────
# Generate the full dataset
# ──────────────────────────────────────────────────────────────────────────
LANGS        = ["en", "tw", "ha", "pcm", "sw"]
TX_TYPES     = ["purchase", "sale", "expense"]
MAKERS       = {"purchase": make_purchase, "sale": make_sale, "expense": make_expense}

# Balanced distribution: 5 langs × 3 types × ~133 = 1995 ≈ 2000
per_bucket   = TOTAL_EXAMPLES // (len(LANGS) * len(TX_TYPES))
extra        = TOTAL_EXAMPLES % (len(LANGS) * len(TX_TYPES))

all_examples: list[dict] = []
for lang in LANGS:
    for tx_type in TX_TYPES:
        n = per_bucket + (1 if extra > 0 else 0)
        extra -= 1
        fn = MAKERS[tx_type]
        for _ in range(n):
            all_examples.append(fn(lang))

random.shuffle(all_examples)

print(f"Generated {len(all_examples)} examples")
# Distribution check
from collections import Counter
lang_counts = Counter(e["_meta"]["lang"] for e in all_examples)
type_counts = Counter(e["_meta"]["type"] for e in all_examples)
print("Language distribution:", dict(lang_counts))
print("Type distribution:    ", dict(type_counts))

# Preview one example from each language
print("\n── Sample examples ──")
seen = set()
for ex in all_examples:
    l = ex["_meta"]["lang"]
    if l not in seen:
        seen.add(l)
        print(f"\n[{l.upper()}] {ex['messages'][1]['content']}")
        print(f"  → {ex['messages'][2]['tool_calls'][0]['function']['name']}(",
              json.dumps(ex['messages'][2]['tool_calls'][0]['function']['arguments'], ensure_ascii=False), ")")
    if len(seen) == len(LANGS):
        break

## 5. Format Dataset for SFT

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Convert tool_calls to the text format Gemma 4 uses natively
# We train on: <start_of_turn>model\n<tool_call>...json...</tool_call>
# matching what Ollama produces when tool calling is active
# ──────────────────────────────────────────────────────────────────────────

def format_tool_call_text(tool_calls: list[dict]) -> str:
    """Render tool_calls as the text format the model learns to produce."""
    parts = []
    for tc in tool_calls:
        fn = tc["function"]
        obj = {"name": fn["name"], "arguments": fn["arguments"]}
        parts.append(json.dumps(obj, ensure_ascii=False))
    return "\n".join(f"<tool_call>\n{p}\n</tool_call>" for p in parts)


def format_for_sft(example: dict) -> dict:
    """
    Convert our message format to a text string for SFT training.
    We format as a Gemma chat turn sequence, making tool calls explicit text.
    """
    msgs = example["messages"]
    system_msg  = msgs[0]["content"]
    user_msg    = msgs[1]["content"]
    tool_calls  = msgs[2]["tool_calls"]

    # Format assistant response as tool call text
    assistant_text = format_tool_call_text(tool_calls)

    # Gemma 4 chat template structure:
    # <start_of_turn>user\n{system}\n\n{user}<end_of_turn>\n
    # <start_of_turn>model\n{assistant}<end_of_turn>
    text = (
        f"<start_of_turn>user\n{system_msg}\n\n{user_msg}<end_of_turn>\n"
        f"<start_of_turn>model\n{assistant_text}<end_of_turn>"
    )

    return {"text": text}


# ── Split train / validation ────────────────────────────────────────────────
val_raw   = all_examples[:VAL_EXAMPLES]
train_raw = all_examples[VAL_EXAMPLES:]

train_formatted = [format_for_sft(e) for e in train_raw]
val_formatted   = [format_for_sft(e) for e in val_raw]

train_dataset = Dataset.from_list(train_formatted)
val_dataset   = Dataset.from_list(val_formatted)

dataset = DatasetDict({"train": train_dataset, "validation": val_dataset})
print(dataset)
print(f"\nTrain examples : {len(train_dataset)}")
print(f"Val examples   : {len(val_dataset)}")
print("\nSample formatted text:")
print(train_dataset[0]["text"][:500])

## 6. Load Gemma 4 E4B with Unsloth

In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name      = MODEL_NAME,
    max_seq_length  = MAX_SEQ_LENGTH,
    dtype           = None,            # auto-detect (bfloat16 on Ampere+, float16 on T4)
    load_in_4bit    = LOAD_IN_4BIT,
    # token = "hf_..."  # HuggingFace token if gated model
)

print("Model loaded:", MODEL_NAME)
print(f"Parameters  : {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B")
print(f"dtype       : {next(model.parameters()).dtype}")

## 7. Configure LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r               = LORA_RANK,
    lora_alpha      = LORA_ALPHA,
    lora_dropout    = LORA_DROPOUT,
    bias            = "none",
    use_gradient_checkpointing = "unsloth",  # saves ~30% VRAM
    random_state    = 42,
    target_modules  = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
)

# Print trainable parameter count
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable parameters : {trainable:,}  ({100*trainable/total:.2f}%)")
print(f"Total parameters     : {total:,}")
print(f"LoRA rank={LORA_RANK}, alpha={LORA_ALPHA}, dropout={LORA_DROPOUT}")

## 8. Fine-Tune with SFTTrainer

In [ ]:
trainer = SFTTrainer(
    model       = model,
    tokenizer   = tokenizer,
    train_dataset = train_dataset,
    eval_dataset  = val_dataset,
    args = SFTConfig(
        dataset_text_field      = "text",
        per_device_train_batch_size = BATCH_SIZE,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio            = WARMUP_RATIO,
        num_train_epochs        = NUM_EPOCHS,
        learning_rate           = LEARNING_RATE,
        fp16                    = not is_bfloat16_supported(),
        bf16                    = is_bfloat16_supported(),
        logging_steps           = 20,
        eval_strategy           = "steps",
        eval_steps              = 100,
        save_strategy           = "steps",
        save_steps              = 100,
        save_total_limit        = 2,
        load_best_model_at_end  = True,
        metric_for_best_model   = "eval_loss",
        optim                   = "adamw_8bit",
        weight_decay            = WEIGHT_DECAY,
        lr_scheduler_type       = LR_SCHEDULER,
        max_grad_norm           = MAX_GRAD_NORM,
        seed                    = 42,
        output_dir              = OUTPUT_DIR,
        report_to               = "none",  # set to "wandb" if you want W&B logging
        max_seq_length          = MAX_SEQ_LENGTH,
    ),
)

# Print GPU memory before training
if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    reserved  = round(torch.cuda.memory_reserved(0) / 1024**3, 2)
    max_mem   = round(gpu_stats.total_memory / 1024**3, 2)
    print(f"GPU: {gpu_stats.name}  VRAM: {max_mem} GB  Reserved: {reserved} GB")

print(f"\nStarting training: {NUM_EPOCHS} epochs × {len(train_dataset)} examples")
print(f"  Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")
print(f"  Steps per epoch    : {math.ceil(len(train_dataset) / (BATCH_SIZE * GRAD_ACCUM))}")

trainer_stats = trainer.train()

In [ ]:
# ── Training summary ────────────────────────────────────────────────────────
elapsed  = trainer_stats.metrics.get("train_runtime", 0)
peak_vram = torch.cuda.max_memory_allocated() / 1e9 if torch.cuda.is_available() else 0

print(f"Training complete!")
print(f"  Runtime        : {elapsed/60:.1f} min")
print(f"  Peak VRAM      : {peak_vram:.2f} GB")
print(f"  Train loss     : {trainer_stats.metrics.get('train_loss', 'N/A'):.4f}")

# Loss curve (if available from logs)
logs = [log for log in trainer.state.log_history if "eval_loss" in log]
if logs:
    print(f"\n  Eval loss history ({len(logs)} checkpoints):")
    for log in logs:
        step   = log.get("step", "?")
        eloss  = log.get("eval_loss", "?")
        print(f"    step {step:4d} → eval_loss {eloss:.4f}")

## 9. Evaluation — Function-Calling Accuracy

In [ ]:
# ──────────────────────────────────────────────────────────────────────────
# Evaluation on the held-out 200 validation examples
# Metrics:
#   1. function_name_acc  — correct function predicted
#   2. param_match_acc    — average % of required params extracted correctly
#   3. exact_match_acc    — all required params exactly right
# ──────────────────────────────────────────────────────────────────────────

FastLanguageModel.for_inference(model)

REQUIRED_PARAMS = {
    "record_purchase": {"item", "quantity", "unit", "currency"},
    "record_sale":     {"item", "quantity", "unit", "currency"},
    "record_expense":  {"description", "amount", "category", "currency"},
}

def parse_tool_call_from_text(text: str) -> dict | None:
    """Parse a <tool_call>...</tool_call> block from model output."""
    pattern = r"<tool_call>\s*({.*?})\s*</tool_call>"
    match   = re.search(pattern, text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group(1))
    except json.JSONDecodeError:
        return None

def params_match_score(pred_args: dict, gold_args: dict, required: set) -> float:
    """Fraction of required params that match between prediction and gold."""
    if not required:
        return 1.0
    correct = 0
    for key in required:
        if key not in gold_args:
            continue
        gold_val = gold_args[key]
        pred_val = pred_args.get(key)
        # Numeric: allow ±1% tolerance
        if isinstance(gold_val, (int, float)):
            try:
                if abs(float(pred_val) - float(gold_val)) / (abs(float(gold_val)) + 1e-6) < 0.01:
                    correct += 1
            except (TypeError, ValueError):
                pass
        # String: case-insensitive, strip underscores/hyphens
        elif isinstance(gold_val, str):
            def norm(s):
                return str(s).lower().replace("_", " ").replace("-", " ").strip()
            if norm(pred_val) == norm(gold_val):
                correct += 1
    return correct / len(required)

# ── Run evaluation ───────────────────────────────────────────────────────────
fn_correct    = 0
exact_correct = 0
param_scores  = []
per_lang: dict[str, list[float]] = defaultdict(list)
per_type: dict[str, list[float]] = defaultdict(list)

eval_inputs = [
    (ex["messages"][1]["content"],
     ex["messages"][2]["tool_calls"][0]["function"]["name"],
     ex["messages"][2]["tool_calls"][0]["function"]["arguments"],
     ex["_meta"]["lang"],
     ex["_meta"]["type"])
    for ex in val_raw
]

BATCH_EVAL = 8   # process in mini-batches to avoid OOM

for start in range(0, len(eval_inputs), BATCH_EVAL):
    batch = eval_inputs[start : start + BATCH_EVAL]

    # Build prompts
    prompts = []
    for user_msg, _, _, _, _ in batch:
        prompt = (f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{user_msg}<end_of_turn>\n"
                  f"<start_of_turn>model\n")
        prompts.append(prompt)

    enc = tokenizer(prompts, return_tensors="pt", padding=True,
                    truncation=True, max_length=MAX_SEQ_LENGTH).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **enc,
            max_new_tokens = 256,
            do_sample      = False,
            temperature    = 1.0,
            pad_token_id   = tokenizer.eos_token_id,
        )

    decoded = tokenizer.batch_decode(outputs[:, enc["input_ids"].shape[1]:],
                                     skip_special_tokens=True)

    for i, (pred_text, (_, gold_fn, gold_args, lang, tx_type)) in enumerate(zip(decoded, batch)):
        parsed = parse_tool_call_from_text(pred_text)

        if parsed is None:
            # Model didn't generate a valid tool call
            param_scores.append(0.0)
            per_lang[lang].append(0.0)
            per_type[tx_type].append(0.0)
            continue

        pred_fn   = parsed.get("name", "")
        pred_args = parsed.get("arguments", {})
        required  = REQUIRED_PARAMS.get(gold_fn, set())

        fn_ok    = (pred_fn == gold_fn)
        score    = params_match_score(pred_args, gold_args, required) if fn_ok else 0.0
        exact_ok = fn_ok and (score == 1.0)

        fn_correct    += fn_ok
        exact_correct += exact_ok
        param_scores.append(score)
        per_lang[lang].append(score)
        per_type[tx_type].append(score)

n = len(eval_inputs)
print(f"\n{'═'*52}")
print(f"  Evaluation Results ({n} examples)")
print(f"{'═'*52}")
print(f"  Function name accuracy : {fn_correct/n*100:.1f}%  ({fn_correct}/{n})")
print(f"  Param match accuracy   : {sum(param_scores)/n*100:.1f}%")
print(f"  Exact match accuracy   : {exact_correct/n*100:.1f}%  ({exact_correct}/{n})")
print(f"{'─'*52}")
print("  Per language:")
for lang, scores in sorted(per_lang.items()):
    avg = sum(scores)/len(scores)*100 if scores else 0
    print(f"    {lang:4s} : {avg:.1f}%  (n={len(scores)})")
print("  Per type:")
for tx_type, scores in sorted(per_type.items()):
    avg = sum(scores)/len(scores)*100 if scores else 0
    print(f"    {tx_type:10s} : {avg:.1f}%  (n={len(scores)})")
print(f"{'═'*52}")

In [ ]:
# ── Qualitative sample predictions ─────────────────────────────────────────
print("\n── Sample Predictions ──\n")
SAMPLE_LANGS = ["en", "tw", "ha", "pcm", "sw"]

for target_lang in SAMPLE_LANGS:
    ex = next(e for e in val_raw if e["_meta"]["lang"] == target_lang)
    user_msg  = ex["messages"][1]["content"]
    gold_fn   = ex["messages"][2]["tool_calls"][0]["function"]["name"]
    gold_args = ex["messages"][2]["tool_calls"][0]["function"]["arguments"]

    prompt = (f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{user_msg}<end_of_turn>\n"
              f"<start_of_turn>model\n")
    enc = tokenizer([prompt], return_tensors="pt", truncation=True,
                    max_length=MAX_SEQ_LENGTH).to(model.device)
    with torch.no_grad():
        out = model.generate(**enc, max_new_tokens=200, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    pred_text = tokenizer.decode(out[0, enc["input_ids"].shape[1]:], skip_special_tokens=True)
    parsed = parse_tool_call_from_text(pred_text)

    print(f"[{target_lang.upper()}]")
    print(f"  Input : {user_msg}")
    print(f"  Gold  : {gold_fn}({json.dumps(gold_args, ensure_ascii=False)})")
    if parsed:
        print(f"  Pred  : {parsed.get('name')}({json.dumps(parsed.get('arguments',{}), ensure_ascii=False)})")
    else:
        print(f"  Pred  : ❌ no tool call found — raw: {pred_text[:120]}")
    print()

## 10. Save LoRA Adapter Weights

In [ ]:
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Save LoRA adapter only (small — ~100 MB)
model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)

print(f"LoRA adapter saved to: {OUTPUT_DIR}")
!du -sh {OUTPUT_DIR}

## 11. Export Merged Model for Ollama

Merge the LoRA weights into the base model and export as GGUF so the
fine-tuned model can be loaded directly into Ollama (same setup as Susu Books uses in production).


In [ ]:
os.makedirs(MERGED_DIR, exist_ok=True)

# ── Merge LoRA into base model ──────────────────────────────────────────────
print("Merging LoRA adapter into base model...")
model.save_pretrained_merged(MERGED_DIR, tokenizer, save_method="merged_16bit")
print(f"Merged model saved: {MERGED_DIR}")
!du -sh {MERGED_DIR}

In [ ]:
# ── Export to GGUF ─────────────────────────────────────────────────────────
GGUF_DIR  = "/kaggle/working/susu-books-gguf"
os.makedirs(GGUF_DIR, exist_ok=True)

print(f"Exporting GGUF with quantisation: {GGUF_QUANT}")
model.save_pretrained_gguf(GGUF_DIR, tokenizer, quantization_method=GGUF_QUANT)

!ls -lh {GGUF_DIR}

In [ ]:
# ── Write Ollama Modelfile ──────────────────────────────────────────────────
import glob
gguf_files = glob.glob(GGUF_DIR + "/*.gguf")
gguf_file  = gguf_files[0] if gguf_files else os.path.join(GGUF_DIR, "model.gguf")
gguf_fname = os.path.basename(gguf_file)

# Build Modelfile content using string concatenation (avoids triple-quote nesting)
lines = [
    "FROM ./" + gguf_fname,
    "",
    "SYSTEM " + chr(34)*3 + SYSTEM_PROMPT + chr(34)*3,
    "",
    "# Susu Books — fine-tuned Gemma 4 E4B",
    "# Optimised for multilingual transaction extraction",
    "# Languages: English, Twi, Hausa, Pidgin English, Swahili",
    "",
    "PARAMETER temperature 0.1",
    "PARAMETER top_p 0.9",
    "PARAMETER repeat_penalty 1.1",
    "PARAMETER num_ctx 4096",
]
modelfile_content = "\n".join(lines)

modelfile_path = os.path.join(GGUF_DIR, "Modelfile")
with open(modelfile_path, "w") as f:
    f.write(modelfile_content)

print("Modelfile written:", modelfile_path)
print()
print("── Deployment instructions ──")
print("1. Download the GGUF + Modelfile from /kaggle/working/susu-books-gguf/")
print("2. On your machine:")
print("     ollama create susu-books-ft -f Modelfile")
print("3. Update backend/.env:")
print("     OLLAMA_MODEL=susu-books-ft")
print("4. Restart: uvicorn main:app --port 8000 --reload")


## 12. Baseline Comparison (Optional)

Run this cell to compare the fine-tuned model against the base Gemma 4 E4B
on the same validation set — shows how much fine-tuning improved accuracy.


In [ ]:
# ── Load base model (no LoRA) and evaluate ─────────────────────────────────
print("Loading base model for comparison...")
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_NAME,
    max_seq_length = MAX_SEQ_LENGTH,
    load_in_4bit   = LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(base_model)

base_exact = 0
base_param_scores = []

for start in range(0, min(100, len(eval_inputs)), BATCH_EVAL):
    batch = eval_inputs[start : start + BATCH_EVAL]
    prompts = [
        f"<start_of_turn>user\n{SYSTEM_PROMPT}\n\n{u}<end_of_turn>\n<start_of_turn>model\n"
        for u, *_ in batch
    ]
    enc = base_tokenizer(prompts, return_tensors="pt", padding=True,
                         truncation=True, max_length=MAX_SEQ_LENGTH).to(base_model.device)
    with torch.no_grad():
        outputs = base_model.generate(**enc, max_new_tokens=256, do_sample=False,
                                      pad_token_id=base_tokenizer.eos_token_id)
    decoded = base_tokenizer.batch_decode(outputs[:, enc["input_ids"].shape[1]:],
                                          skip_special_tokens=True)
    for pred_text, (_, gold_fn, gold_args, lang, tx_type) in zip(decoded, batch):
        parsed  = parse_tool_call_from_text(pred_text)
        if parsed:
            fn_ok  = parsed.get("name") == gold_fn
            score  = params_match_score(parsed.get("arguments",{}), gold_args,
                                        REQUIRED_PARAMS.get(gold_fn, set())) if fn_ok else 0.0
            base_exact += fn_ok and score == 1.0
            base_param_scores.append(score)
        else:
            base_param_scores.append(0.0)

n_base = len(base_param_scores)
print(f"\n{'═'*52}")
print(f"  Baseline vs Fine-Tuned (first {n_base} val examples)")
print(f"{'═'*52}")
print(f"  {'Metric':<30} {'Base':>8} {'FT':>8}")
print(f"  {'─'*46}")
print(f"  {'Param match accuracy':<30} {sum(base_param_scores)/n_base*100:>7.1f}% {sum(param_scores[:n_base])/n_base*100:>7.1f}%")
print(f"  {'Exact match accuracy':<30} {base_exact/n_base*100:>7.1f}% {sum(1 for s in param_scores[:n_base] if s==1.0)/n_base*100:>7.1f}%")
print(f"{'═'*52}")

del base_model, base_tokenizer
torch.cuda.empty_cache()

## Summary

| Step | Result |
|------|--------|
| **Data** | 2 000 synthetic examples across 5 languages × 3 transaction types |
| **Model** | Gemma 4 E4B fine-tuned with LoRA (r=16, α=32) |
| **Training** | 3 epochs, lr=2e-4, effective batch=16 |
| **Output** | LoRA adapter + merged GGUF (Q4_K_M) for Ollama |

### Next steps
1. Download `susu-books-gguf/` from Kaggle output
2. `ollama create susu-books-ft -f Modelfile`
3. Set `OLLAMA_MODEL=susu-books-ft` in `backend/.env`
4. Re-run `setup.sh` — the app will now use the fine-tuned model

The fine-tuned model should show improved accuracy on:
- Code-switched utterances (mixing local language + English)
- Numeric extraction (words vs digits)
- Local item name variants (shinkafa, nkuto, albasa → rice, palm_oil, onions)
- Informal expense descriptions common in West African markets

---
*Built for the Gemma 4 Good Hackathon · Kaggle 2026*
*Repository: [Susu Books](https://github.com/YOUR_USERNAME/susu-books)*
